# Divvy Bikeshare Data Lake Project

## Overview
This project implements a Lakehouse architecture using Azure Databricks and Delta Lake.

## Business Requirements
- Analyze ride duration
- Analyze usage by time
- Analyze usage by station
- Analyze usage by rider age
- Analyze revenue

In [0]:
df_trip_raw = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .option("sep", ",") \
    .load("dbfs:/FileStore/tables/trips.csv")

df_trip_raw.show(3)

+----------------+-------------+-------------------+-------------------+------------+------------+-----+
|             _c0|          _c1|                _c2|                _c3|         _c4|         _c5|  _c6|
+----------------+-------------+-------------------+-------------------+------------+------------+-----+
|89E7AA6C29227EFF| classic_bike|2021-02-12 16:14:56|2021-02-12 16:21:43|         525|         660|71934|
|0FEFDE2603568365| classic_bike|2021-02-14 17:52:38|2021-02-14 18:12:09|         525|       16806|47854|
|E6159D746B2DBB91|electric_bike|2021-02-09 19:10:18|2021-02-09 19:19:10|KA1503000012|TA1305000029|70870|
+----------------+-------------+-------------------+-------------------+------------+------------+-----+
only showing top 3 rows


In [0]:
df_trip = df_trip_raw.toDF(
    "ride_id",
    "rideable_type",
    "started_at",
    "ended_at",
    "start_station_id",
    "end_station_id",
    "rider_id"
)

df_trip.show(3)

+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|         ride_id|rideable_type|         started_at|           ended_at|start_station_id|end_station_id|rider_id|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|89E7AA6C29227EFF| classic_bike|2021-02-12 16:14:56|2021-02-12 16:21:43|             525|           660|   71934|
|0FEFDE2603568365| classic_bike|2021-02-14 17:52:38|2021-02-14 18:12:09|             525|         16806|   47854|
|E6159D746B2DBB91|electric_bike|2021-02-09 19:10:18|2021-02-09 19:19:10|    KA1503000012|  TA1305000029|   70870|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
only showing top 3 rows


In [0]:
df_trip.write.format("delta") \
    .mode("overwrite") \
    .save("/delta/bronze/trip")


spark.read.format("delta") \
    .load("/delta/bronze/trip") \
    .show(3)

+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|         ride_id|rideable_type|         started_at|           ended_at|start_station_id|end_station_id|rider_id|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
|7458DFFCC69A9C21| classic_bike|2021-11-06 16:53:22|2021-11-06 17:18:20|    TA1309000063|  TA1307000048|    2830|
|507FB5FF150F9004|electric_bike|2021-11-24 06:45:31|2021-11-24 06:50:43|           13258|         13084|   40256|
|3E2475AC2E54C0A2| classic_bike|2021-11-30 08:07:37|2021-11-30 08:14:57|           13258|         13084|   35689|
+----------------+-------------+-------------------+-------------------+----------------+--------------+--------+
only showing top 3 rows


In [0]:
dbutils.fs.ls("dbfs:/FileStore/tables/")

[FileInfo(path='dbfs:/FileStore/tables/divvy/', name='divvy/', size=0, modificationTime=1776430424000),
 FileInfo(path='dbfs:/FileStore/tables/payments.csv', name='payments.csv', size=57666115, modificationTime=1776431122000),
 FileInfo(path='dbfs:/FileStore/tables/riders.csv', name='riders.csv', size=5594949, modificationTime=1776431100000),
 FileInfo(path='dbfs:/FileStore/tables/stations.csv', name='stations.csv', size=49552, modificationTime=1776430426000),
 FileInfo(path='dbfs:/FileStore/tables/trips.csv', name='trips.csv', size=440125504, modificationTime=1776430602000)]

In [0]:
# =========================
# PAYMENT
# =========================
df_payment_raw = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .option("sep", ",") \
    .load("dbfs:/FileStore/tables/payments.csv")

df_payment = df_payment_raw.toDF(
    "payment_id",
    "date",
    "amount",
    "account_number"
)

df_payment.write.format("delta").mode("overwrite").save("/delta/bronze/payment")

spark.read.format("delta").load("/delta/bronze/payment").show(3)


# =========================
# STATION
# =========================
df_station_raw = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .option("sep", ",") \
    .load("dbfs:/FileStore/tables/stations.csv")

df_station = df_station_raw.toDF(
    "station_id",
    "name",
    "latitude",
    "longitude"
)

df_station.write.format("delta").mode("overwrite").save("/delta/bronze/station")

spark.read.format("delta").load("/delta/bronze/station").show(3)


# =========================
# RIDER
# =========================
df_rider_raw = spark.read.format("csv") \
    .option("inferSchema", "true") \
    .option("header", "false") \
    .option("sep", ",") \
    .load("dbfs:/FileStore/tables/riders.csv")

df_rider = df_rider_raw.toDF(
    "rider_id",
    "first",
    "last",
    "address",
    "birthday",
    "account_start_date",
    "account_end_date",
    "is_member"
)

df_rider.write.format("delta").mode("overwrite").save("/delta/bronze/rider")

spark.read.format("delta").load("/delta/bronze/rider").show(3)

+----------+----------+------+--------------+
|payment_id|      date|amount|account_number|
+----------+----------+------+--------------+
|         1|2019-05-01|   9.0|          1000|
|         2|2019-06-01|   9.0|          1000|
|         3|2019-07-01|   9.0|          1000|
+----------+----------+------+--------------+
only showing top 3 rows
+------------+--------------------+-----------------+------------------+
|  station_id|                name|         latitude|         longitude|
+------------+--------------------+-----------------+------------------+
|         525|Glenwood Ave & To...|        42.012701|-87.66605799999999|
|KA1503000012|  Clark St & Lake St|41.88579466666667|-87.63110066666668|
|         637|Wood St & Chicago...|        41.895634|        -87.672069|
+------------+--------------------+-----------------+------------------+
only showing top 3 rows
+--------+--------+-----+--------------------+----------+------------------+----------------+---------+
|rider_id|   fi

In [0]:
from pyspark.sql.functions import col, unix_timestamp

df_trip = spark.read.format("delta").load("/delta/bronze/trip")

df_trip_silver = df_trip \
    .withColumn("started_at", col("started_at").cast("timestamp")) \
    .withColumn("ended_at", col("ended_at").cast("timestamp")) \
    .withColumn(
        "trip_duration_minutes",
        (unix_timestamp("ended_at") - unix_timestamp("started_at")) / 60
    )

df_trip_silver.write.format("delta").mode("overwrite").save("/delta/silver/trip")

spark.read.format("delta") \
    .load("/delta/silver/trip") \
    .select("ride_id", "trip_duration_minutes") \
    .show(3)

+----------------+---------------------+
|         ride_id|trip_duration_minutes|
+----------------+---------------------+
|2B065A1F466040A6|                 13.9|
|2480CCBA8E545AF1|                  8.0|
|01694EAF7FD14738|    7.033333333333333|
+----------------+---------------------+
only showing top 3 rows


In [0]:
from pyspark.sql.functions import year, col

df_trip_silver = spark.read.format("delta").load("/delta/silver/trip").alias("t")
df_rider = spark.read.format("delta").load("/delta/bronze/rider").alias("r")

df_trip_silver = df_trip_silver.join(
    df_rider.select("rider_id", "birthday").alias("r"),
    col("t.rider_id") == col("r.rider_id"),
    "left"
).withColumn(
    "age_at_trip",
    year(col("t.started_at")) - year(col("r.birthday"))
)

df_trip_silver = df_trip_silver.select(
    "t.*",
    "age_at_trip"
)

df_trip_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/delta/silver/trip")

spark.read.format("delta") \
    .load("/delta/silver/trip") \
    .select("rider_id", "age_at_trip") \
    .show(3)

+--------+-----------+
|rider_id|age_at_trip|
+--------+-----------+
|   71934|         38|
|   47854|         39|
|   70870|         34|
+--------+-----------+
only showing top 3 rows


In [0]:
from pyspark.sql.functions import to_date, year, month, quarter, dayofweek

# carregar dados
df_trip = spark.read.format("delta").load("/delta/silver/trip")
df_payment = spark.read.format("delta").load("/delta/bronze/payment")

# unir datas de trip + payment
df_dates = df_trip.select(to_date("started_at").alias("date")) \
    .union(
        df_payment.select(to_date("date").alias("date"))
    ).distinct()

# criar dimensão
df_date = df_dates \
    .withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("quarter", quarter("date")) \
    .withColumn("day_of_week", dayofweek("date"))

# salvar COMO TABELA (já antecipando próximo erro)
df_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_date")

# validação
spark.sql("SELECT * FROM dim_date LIMIT 5").show()

+----------+----+-----+-------+-----------+
|      date|year|month|quarter|day_of_week|
+----------+----+-----+-------+-----------+
|2021-12-05|2021|   12|      4|          1|
|2021-12-02|2021|   12|      4|          5|
|2021-09-06|2021|    9|      3|          2|
|2021-10-31|2021|   10|      4|          1|
|2021-10-19|2021|   10|      4|          3|
+----------+----+-----+-------+-----------+



In [0]:
from pyspark.sql.functions import year

df_rider = spark.read.format("delta").load("/delta/bronze/rider")

df_rider_dim = df_rider.select(
    "rider_id",
    "is_member",
    "birthday",
    "account_start_date"
).withColumn(
    "age_at_account_start",
    year("account_start_date") - year("birthday")
)

df_rider_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_rider")

# validação
spark.sql("""
SELECT rider_id, is_member, age_at_account_start 
FROM dim_rider 
LIMIT 5
""").show()

+--------+---------+--------------------+
|rider_id|is_member|age_at_account_start|
+--------+---------+--------------------+
|    1000|     true|                  30|
|    1001|     true|                  43|
|    1002|     true|                  24|
|    1003|    false|                  20|
|    1004|     true|                  50|
+--------+---------+--------------------+



In [0]:
spark.sql("SHOW TABLES").show()

+--------+------------+-----------+
|database|   tableName|isTemporary|
+--------+------------+-----------+
| default|    dim_date|      false|
| default|   dim_rider|      false|
| default| dim_station|      false|
| default|fact_payment|      false|
| default|   fact_trip|      false|
+--------+------------+-----------+



In [0]:
df_station = spark.read.format("delta").load("/delta/bronze/station")

df_station_dim = df_station.select(
    "station_id",
    "name",
    "latitude",
    "longitude"
)

df_station_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_station")

# validação
spark.sql("""
SELECT * 
FROM dim_station 
LIMIT 5
""").show()

+------------+--------------------+-----------------+------------------+
|  station_id|                name|         latitude|         longitude|
+------------+--------------------+-----------------+------------------+
|         525|Glenwood Ave & To...|        42.012701|-87.66605799999999|
|KA1503000012|  Clark St & Lake St|41.88579466666667|-87.63110066666668|
|         637|Wood St & Chicago...|        41.895634|        -87.672069|
|       13216|  State St & 33rd St|       41.8347335|       -87.6258275|
|       18003|Fairbanks St & Su...|41.89580766666667|-87.62025316666669|
+------------+--------------------+-----------------+------------------+



In [0]:
from pyspark.sql.functions import to_date

# carregar silver
df_trip_silver = spark.read.format("delta").load("/delta/silver/trip")

# criar fact
df_fact_trip = df_trip_silver.select(
    "ride_id",
    "rider_id",
    "start_station_id",
    "end_station_id",
    to_date("started_at").alias("date"),
    "trip_duration_minutes",
    "age_at_trip"
)

# salvar COMO TABELA (CORREÇÃO PRINCIPAL)
df_fact_trip.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_trip")

# validação (usar SQL agora)
spark.sql("""
SELECT ride_id, trip_duration_minutes, age_at_trip 
FROM fact_trip 
LIMIT 5
""").show()

+----------------+---------------------+-----------+
|         ride_id|trip_duration_minutes|age_at_trip|
+----------------+---------------------+-----------+
|05961376528D5AA9|    8.183333333333334|         22|
|042029FFA981571B|   16.166666666666668|         26|
|F0E9906B452FC22A|   20.783333333333335|         42|
|D5548D6B05F9DB39|   30.616666666666667|         37|
|0CDF8C5894FDCAAB|                 8.65|         35|
+----------------+---------------------+-----------+



In [0]:
df_payment = spark.read.format("delta").load("/delta/bronze/payment")

df_fact_payment = df_payment.select(
    "payment_id",
    "account_number",  # mantém aqui (dado original)
    "date",
    "amount"
)

df_fact_payment.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fact_payment")

# validação
spark.sql("""
SELECT payment_id, amount 
FROM fact_payment 
LIMIT 5
""").show()

+----------+------+
|payment_id|amount|
+----------+------+
|         1|   9.0|
|         2|   9.0|
|         3|   9.0|
|         4|   9.0|
|         5|   9.0|
+----------+------+



In [0]:
spark.read.format("delta").load("/delta/gold/fact_trip").count()
spark.read.format("delta").load("/delta/gold/fact_payment").count()

1946607

In [0]:
spark.read.format("delta").load("/delta/gold/fact_trip").select("ride_id").distinct().count()

spark.read.format("delta").load("/delta/gold/fact_payment").select("payment_id").distinct().count()

1946607

Although Fact_Trip and Fact_Payment have similar row counts, they represent different business processes:

- Fact_Trip captures ride behavior (duration, usage)
- Fact_Payment captures financial transactions (amount spent)

This separation follows dimensional modeling best practices.